<!-- paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 참고 구현입니다. 필요한 한 줄만 확인하고 다시 실습 창으로 돌아가세요.

# 20. 표형 MLP와 비지도 센서 이상 탐지

희소한 계정 위험 레이블을 MLP로 분류하고 threshold를 validation에서 선택합니다.
이어서 센서 정상 구간만으로 autoencoder를 학습해 reconstruction error 기반 이상 탐지를
구성합니다. 지도/비지도 접근의 데이터 요구 사항과 평가 차이를 비교합니다.

In [ ]:
# 이 컴퓨터를 감지해 안전한 시작값을 확인합니다. 각 실습은 더 작은 값으로도 동작합니다.
from llm_engineering_lab.hardware import HardwareProfile, recommend_training_profile

LAB_HARDWARE = HardwareProfile.detect()
LAB_PROFILE = recommend_training_profile(LAB_HARDWARE)
print("hardware:", LAB_HARDWARE)
print("recommended profile:", LAB_PROFILE)

## 학습 목표

- train 통계만 사용하는 숫자 표준화와 범주 one-hot 파이프라인을 만든다.
- `pos_weight`로 class imbalance를 손실에 반영한다.
- 0.5 고정 대신 validation F1로 threshold를 고른다.
- 정상 데이터만으로 autoencoder를 fit하고 validation error로 임계값을 정한다.
- accuracy가 희소 이벤트에서 왜 위험한 지표인지 설명한다.

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from llm_engineering_lab.acceleration import get_accelerator

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
ACCELERATOR = get_accelerator()
device = ACCELERATOR.device
print(ACCELERATOR.summary())

In [ ]:
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "practice").exists():
            return candidate
    raise FileNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. Start_JupyterLab.cmd로 실행해 주세요."
    )


ROOT = find_project_root()
DATA = ROOT / "data" / "practice"
ARTIFACTS = ROOT / "artifacts" / "practice"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("project root:", ROOT)

## 1) 희소한 계정 risk 데이터

accuracy만 보면 모든 샘플을 정상으로 예측하는 모델도 높아 보일 수 있습니다.
positive 비율과 naive baseline을 먼저 계산한 뒤 precision/recall/F1을 사용합니다.

In [ ]:
frame = pd.read_csv(DATA / "tabular_risk.csv")
target = "risk_label"
identifier = "account_id"
print(frame.shape, frame.dtypes.to_dict())
positive_rate = frame[target].mean()
print(
    "positive rate:",
    round(positive_rate, 4),
    "all-negative accuracy:",
    round(1 - positive_rate, 4),
)

In [ ]:
train_frame, temp_frame = train_test_split(
    frame, test_size=0.30, random_state=SEED, stratify=frame[target]
)
valid_frame, test_frame = train_test_split(
    temp_frame, test_size=0.50, random_state=SEED, stratify=temp_frame[target]
)
numeric = [
    "tenure_months",
    "monthly_spend",
    "weekly_sessions",
    "support_tickets_90d",
    "latency_ms",
]
categorical = ["industry", "region", "tier"]
preprocessor = ColumnTransformer(
    [
        (
            "numeric",
            Pipeline(
                [
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]
            ),
            numeric,
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical,
        ),
    ]
)
X_train = preprocessor.fit_transform(train_frame).astype(np.float32)
X_valid = preprocessor.transform(valid_frame).astype(np.float32)
X_test = preprocessor.transform(test_frame).astype(np.float32)
y_train = train_frame[target].to_numpy(np.float32)
y_valid = valid_frame[target].to_numpy(np.int64)
y_test = test_frame[target].to_numpy(np.int64)
print(X_train.shape, X_valid.shape, X_test.shape)

In [ ]:
assert X_train.dtype == X_valid.dtype == X_test.dtype == np.float32
assert X_train.shape[1] == X_valid.shape[1] == X_test.shape[1]
assert (
    np.isfinite(X_train).all()
    and np.isfinite(X_valid).all()
    and np.isfinite(X_test).all()
)
assert abs(y_train.mean() - y_test.mean()) < 0.03
print("preprocessing contract: OK")

## 2) class imbalance를 고려한 MLP

`BCEWithLogitsLoss(pos_weight=negative/positive)`는 양성 오류를 더 크게 벌점화합니다.
확률을 먼저 sigmoid한 뒤 BCE를 적용하지 말고, 수치적으로 안정적인 logits 손실을 사용합니다.

In [ ]:
class RiskMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=32):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x):
        return self.network(x).squeeze(1)


model = RiskMLP(X_train.shape[1]).to(device)
positives = float(y_train.sum())
pos_weight = torch.tensor([(len(y_train) - positives) / positives], device=device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
print("pos_weight:", pos_weight.item())

In [ ]:
train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
    batch_size=64,
    shuffle=True,
)
print(
    "batches:",
    len(train_loader),
    "parameters:",
    sum(p.numel() for p in model.parameters()),
)

In [ ]:
for epoch in range(10):
    model.train()
    total = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(y)
    if epoch in {0, 4, 9}:
        print(f"epoch={epoch + 1:02d} loss={total / len(train_loader.dataset):.4f}")


@torch.inference_mode()
def predict_probability(model, features):
    model.eval()
    x = torch.from_numpy(features).to(device)
    return model(x).sigmoid().cpu().numpy()


valid_probability = predict_probability(model, X_valid)

## 3) threshold는 validation에서 선택

손실의 class weight와 의사결정 threshold는 별개입니다. 운영 비용에 따라 precision과
recall의 선호가 달라집니다. 여기서는 단순화를 위해 validation F1 최대값을 선택합니다.

In [ ]:
thresholds = np.linspace(0.10, 0.90, 81)
valid_f1 = np.array(
    [
        f1_score(y_valid, valid_probability >= threshold, zero_division=0)
        for threshold in thresholds
    ]
)
best_threshold = float(thresholds[valid_f1.argmax()])
print("best threshold:", best_threshold, "validation F1:", valid_f1.max())

In [ ]:
test_probability = predict_probability(model, X_test)
test_prediction = test_probability >= best_threshold
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, test_prediction, average="binary", zero_division=0
)
auc = roc_auc_score(y_test, test_probability)
supervised_metrics = {
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "roc_auc": auc,
}
print(supervised_metrics)
assert all(np.isfinite(value) for value in supervised_metrics.values())

## 4) 정상 센서만 학습하는 autoencoder

레이블이 부족할 때는 정상 패턴을 압축·복원하도록 학습하고 복원 오차가 큰 샘플을
이상으로 간주할 수 있습니다. train에는 시간상 앞부분의 정상 행만 사용합니다.

In [ ]:
sensor = pd.read_csv(DATA / "sensor_timeseries.csv", parse_dates=["timestamp"])
sensor_features = ["load", "temperature", "vibration", "pressure"]
ordered = sensor.sort_values(["device_id", "timestamp"]).reset_index(drop=True)
train_parts, valid_parts, test_parts = [], [], []
for _, group in ordered.groupby("device_id", sort=False):
    a, b = int(len(group) * 0.60), int(len(group) * 0.80)
    train_parts.append(group.iloc[:a])
    valid_parts.append(group.iloc[a:b])
    test_parts.append(group.iloc[b:])
ae_train = pd.concat(train_parts)
ae_valid = pd.concat(valid_parts)
ae_test = pd.concat(test_parts)
ae_means = ae_train[sensor_features].mean()
ae_stds = ae_train[sensor_features].std().replace(0, 1)
normal_train = ae_train.query("anomaly == 0")
print(
    len(normal_train),
    len(ae_valid),
    len(ae_test),
    "test anomalies:",
    int(ae_test.anomaly.sum()),
)

In [ ]:
def sensor_matrix(frame):
    return ((frame[sensor_features] - ae_means) / ae_stds).to_numpy(np.float32)


ae_train_x = sensor_matrix(normal_train)
ae_valid_x = sensor_matrix(ae_valid)
ae_test_x = sensor_matrix(ae_test)


class AutoEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 8), nn.ReLU(), nn.Linear(8, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 8), nn.ReLU(), nn.Linear(8, input_dim)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


autoencoder = AutoEncoder(len(sensor_features)).to(device)
ae_optimizer = torch.optim.Adam(autoencoder.parameters(), lr=3e-3)
ae_loader = DataLoader(
    TensorDataset(torch.from_numpy(ae_train_x)), batch_size=128, shuffle=True
)

In [ ]:
for epoch in range(10):
    autoencoder.train()
    total = 0.0
    for (x,) in ae_loader:
        x = x.to(device)
        ae_optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.mse_loss(autoencoder(x), x)
        loss.backward()
        ae_optimizer.step()
        total += loss.item() * len(x)
    if epoch in {0, 4, 9}:
        print(f"AE epoch={epoch + 1:02d} loss={total / len(ae_loader.dataset):.4f}")


@torch.inference_mode()
def reconstruction_error(features):
    autoencoder.eval()
    x = torch.from_numpy(features).to(device)
    return ((autoencoder(x) - x) ** 2).mean(dim=1).cpu().numpy()


valid_error = reconstruction_error(ae_valid_x)
test_error = reconstruction_error(ae_test_x)

## 5) 정상 validation 오차로 anomaly threshold 설정

threshold 선택에도 test label을 사용하면 평가가 낙관적으로 변합니다. validation 중
정상 샘플 오차의 99% 분위수를 사용하고 test label은 마지막 평가에만 사용합니다.

In [ ]:
valid_normal_error = valid_error[ae_valid.anomaly.to_numpy() == 0]
anomaly_threshold = float(np.quantile(valid_normal_error, 0.99))
anomaly_prediction = test_error >= anomaly_threshold
anomaly_true = ae_test.anomaly.to_numpy()
ae_precision, ae_recall, ae_f1, _ = precision_recall_fscore_support(
    anomaly_true, anomaly_prediction, average="binary", zero_division=0
)
ae_auc = roc_auc_score(anomaly_true, test_error)
unsupervised_metrics = {
    "threshold": anomaly_threshold,
    "precision": ae_precision,
    "recall": ae_recall,
    "f1": ae_f1,
    "roc_auc": ae_auc,
}
print(unsupervised_metrics)

In [ ]:
assert anomaly_threshold > 0
assert all(np.isfinite(v) for v in unsupervised_metrics.values())
print(
    "detected/actual anomalies:", int(anomaly_prediction.sum()), int(anomaly_true.sum())
)

## 지도 분류와 비지도 이상 탐지 비교

| 방식 | 학습 데이터 | score | 장점 | 주의점 |
|---|---|---|---|---|
| MLP | 정상+위험 label | risk probability | 목표 label에 직접 최적화 | label 비용, 분포 변화 |
| Autoencoder | 주로 정상 | reconstruction error | 이상 label 없이 시작 | 잘 복원되는 이상, threshold 민감도 |

두 결과의 수치는 데이터셋과 target 정의가 달라 직접적인 승패 비교가 아닙니다.

In [ ]:
comparison = pd.DataFrame(
    [
        {"task": "account risk / supervised MLP", **supervised_metrics},
        {
            "task": "sensor anomaly / autoencoder",
            **{k: v for k, v in unsupervised_metrics.items() if k != "threshold"},
        },
    ]
).set_index("task")
print(comparison)

## 운영 threshold 실험

false negative 비용이 크면 threshold를 낮춰 recall을 높이고, 검토 인력이 제한되면
precision 또는 상위 K alert를 최적화할 수 있습니다. 여러 threshold의 alert 수와
precision/recall을 표로 만들어 운영팀과 합의하세요.

In [ ]:
operating_points = []
for threshold in (0.3, 0.5, best_threshold, 0.7):
    prediction = test_probability >= threshold
    p, r, score, _ = precision_recall_fscore_support(
        y_test, prediction, average="binary", zero_division=0
    )
    operating_points.append(
        {
            "threshold": round(float(threshold), 3),
            "alerts": int(prediction.sum()),
            "precision": p,
            "recall": r,
            "f1": score,
        }
    )
print(pd.DataFrame(operating_points).drop_duplicates("threshold"))

## 마무리

불균형 문제에서는 모델 구조보다 split, train-only fit, 손실 가중치, validation threshold,
운영 비용의 정의가 성능을 좌우합니다. autoencoder는 좋은 시작점이지만 시계열 문맥까지
필요하다면 sequence autoencoder나 forecasting residual로 확장해 보세요.